# TEMA with volatility-based position sizing

A triple exponential moving average strategy with volatility-based sizing, execution controls, and a buy-and-hold comparison.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
TRIPLE EXPONENTIAL MOVING AVERAGE (TEMA) LONG-SHORT STRATEGY
===========================================================

STRATEGY OVERVIEW:
-----------------
This strategy uses the Triple Exponential Moving Average (TEMA) as a trend-following
indicator to determine market direction. The strategy is ALWAYS in the market,
switching between LONG and SHORT positions based on TEMA direction.

TEMA ADVANTAGES:
- Reduced lag compared to simple EMA
- Smoother trend identification
- Better responsiveness to price changes

TRADING RULES:
-------------
SIGNAL GENERATION (at Close[t]):
- If TEMA[t] > TEMA[t-1]: Signal = LONG (trend is rising)
- If TEMA[t] < TEMA[t-1]: Signal = SHORT (trend is falling)

EXECUTION (at Open[t+1]):
- Enter LONG at Open[t+1] if signal is LONG
- Enter SHORT at Open[t+1] if signal is SHORT
- Always 100% invested (either LONG or SHORT)

RETURN CALCULATION:
- On execution day: Open[t+1]/Open[t] - 1
- During holding: Close[t]/Close[t-1] - 1

NO LOOK-AHEAD BIAS:
- TEMA calculated using only historical data up to t-1
- Signals generated at Close[t] using data through t-1
- Execution occurs at Open[t+1]
"""

from _market_data import download_ohlcv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple, Optional
from dataclasses import dataclass

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class StrategyConfig:
    """
    Centralized configuration for all strategy parameters.
    
    This ensures all parameters are easily modifiable and documented.
    NO HARDCODED VALUES elsewhere in the code.
    """
    
    # API Configuration
    
    # Asset Selection
    TICKER: str = "TSLA"  # Primary asset to trade
    BENCHMARK_TICKER: str = "URTH"  # Global equity benchmark
    
    # Date Range (5 years ending 2024-12-31)
    END_DATE: str = "2024-12-31"
    START_DATE: str = "2020-01-01"  
    
    # TEMA Parameters
    TEMA_PERIOD: int = 20  # Lookback period for TEMA calculation
    
    # Trading Parameters
    INITIAL_CAPITAL: float = 100000.0
    POSITION_SIZE: float = 1.0  # Always 100% invested (long or short)
    
    # Transaction Costs (realistic institutional costs)
    COMMISSION_PER_TRADE: float = 1.0  # Fixed commission per trade
    COMMISSION_PCT: float = 0.0005  # 5 basis points (0.05%)
    BID_ASK_SPREAD: float = 0.0003  # 3 basis points
    MARKET_IMPACT: float = 0.0002  # 2 basis points
    
    # Short Position Costs
    SHORT_BORROW_RATE: float = 0.03  # 3% annual cost to borrow shares
    
    # Risk-Free Rate
    RISK_FREE_RATE: float = 0.02  # 2% annual for Sharpe calculation


# ============================================================================
# DATA ACQUISITION MODULE
# ============================================================================

class DataFetcher:
    """
    Handles data acquisition from Yahoo Finance with comprehensive quality checks.
    
    CRITICAL: No forward-filling or back-filling of data.
    We use ONLY the data that exists to maintain integrity.
    """
    
    
    def fetch_ohlcv_data(self, ticker: str, start_date: str, end_date: str, buffer_days: int = 100) -> pd.DataFrame:
        buffer_start = pd.Timestamp(start_date) - pd.Timedelta(days=buffer_days)
        frame = download_ohlcv(ticker, buffer_start, end_date).rename_axis("Date").reset_index()
        min_required = buffer_days + 20
        if len(frame) < min_required:
            raise ValueError(f"Only {len(frame)} rows for {ticker}; need at least {min_required}.")
        invalid_ohlc = (
            (frame["Low"] > frame["Open"]) | (frame["Low"] > frame["Close"])
            | (frame["High"] < frame["Open"]) | (frame["High"] < frame["Close"])
        ).sum()
        if invalid_ohlc > len(frame) * 0.01:
            raise ValueError(f"Invalid OHLC values in {invalid_ohlc} rows for {ticker}.")
        if (frame[["Open", "Close"]] <= 0).any().any():
            raise ValueError(f"Nonpositive prices returned for {ticker}.")
        return frame[["Date", "Open", "High", "Low", "Close", "Volume"]]


# ============================================================================
# TEMA CALCULATION MODULE
# ============================================================================

class TEMACalculator:
    """
    Calculate Triple Exponential Moving Average (TEMA).
    
    TEMA FORMULA:
    =============
    EMA1 = EMA(price, period)
    EMA2 = EMA(EMA1, period)
    EMA3 = EMA(EMA2, period)
    TEMA = 3*EMA1 - 3*EMA2 + EMA3
    
    This triple smoothing reduces lag while maintaining smoothness.
    
    NO LOOK-AHEAD BIAS:
    ===================
    All EMA calculations use only historical data. At time t,
    we only know prices up to t-1, ensuring no future data contamination.
    """
    
    @staticmethod
    def calculate_ema(prices: pd.Series, period: int) -> pd.Series:
        """
        Calculate Exponential Moving Average.
        
        EMA uses exponential weighting, giving more weight to recent prices:
        EMA[t] = Price[t] * k + EMA[t-1] * (1-k)
        where k = 2 / (period + 1)
        
        Args:
            prices: Price series
            period: EMA period
            
        Returns:
            EMA series
        """
        return prices.ewm(span=period, adjust=False, min_periods=period).mean()
    
    @classmethod
    def calculate_tema(cls, prices: pd.Series, period: int) -> pd.Series:
        """
        Calculate Triple Exponential Moving Average.
        
        Process:
        1. Calculate first EMA of prices
        2. Calculate second EMA of first EMA
        3. Calculate third EMA of second EMA
        4. Combine: TEMA = 3*EMA1 - 3*EMA2 + EMA3
        
        Args:
            prices: Close price series
            period: TEMA period (e.g., 20)
            
        Returns:
            TEMA series with NaN for insufficient data points
        """
        # Step 1: First EMA of prices
        ema1 = cls.calculate_ema(prices, period)
        
        # Step 2: Second EMA (EMA of EMA1)
        ema2 = cls.calculate_ema(ema1, period)
        
        # Step 3: Third EMA (EMA of EMA2)
        ema3 = cls.calculate_ema(ema2, period)
        
        # Step 4: Combine using TEMA formula
        # TEMA = 3*EMA1 - 3*EMA2 + EMA3
        tema = 3 * ema1 - 3 * ema2 + ema3
        
        return tema
    
    @classmethod
    def calculate_tema_with_validation(cls,
                                      df: pd.DataFrame,
                                      period: int,
                                      price_col: str = 'Close') -> pd.DataFrame:
        """
        Calculate TEMA and add to DataFrame with validation.
        
        CRITICAL: TEMA is calculated using Close[t-1] for signal at t.
        This ensures we only use available data when generating signals.
        
        Args:
            df: DataFrame with price data
            period: TEMA period
            price_col: Column name for prices (default: 'Close')
            
        Returns:
            DataFrame with TEMA column added
        """
        df = df.copy()
        
        # Calculate TEMA from Close prices
        df['TEMA'] = cls.calculate_tema(df[price_col], period)
        
        # LAG TEMA by 1 period to ensure T-1 data for T signals
        # At time T, we only know TEMA through T-1
        df['TEMA_lag'] = df['TEMA'].shift(1)
        
        # Calculate TEMA change (for signal generation)
        # TEMA is rising if TEMA[t-1] > TEMA[t-2]
        df['TEMA_change'] = df['TEMA_lag'].diff()
        
        # Count valid TEMA values
        valid_count = df['TEMA_lag'].notna().sum()
        total_count = len(df)
        
        print(f"\nTEMA Calculation:")
        print(f"  Period: {period}")
        print(f"  Valid TEMA values: {valid_count}/{total_count} ({valid_count/total_count:.1%})")
        print(f"  Warm-up period: ~{period * 3} days (3x period for triple smoothing)")
        
        # Validate TEMA calculation
        if valid_count < 50:
            raise ValueError(
                f"Insufficient valid TEMA values: {valid_count}. "
                f"Need more historical data."
            )
        
        return df


# ============================================================================
# SIGNAL GENERATION MODULE
# ============================================================================

class SignalGenerator:
    """
    Generate trading signals based on TEMA direction.
    
    SIGNAL LOGIC:
    =============
    At Close[t], we observe:
    - TEMA[t-1] (lagged TEMA, known at t)
    - TEMA[t-2] (previous lagged TEMA)
    
    Signal Rules:
    - If TEMA[t-1] > TEMA[t-2]: Signal = LONG (trend rising)
    - If TEMA[t-1] < TEMA[t-2]: Signal = SHORT (trend falling)
    - If TEMA[t-1] == TEMA[t-2]: Keep previous signal (no change)
    
    Execution: Signal generated at Close[t] executes at Open[t+1]
    """
    
    @staticmethod
    def generate_signals(df: pd.DataFrame) -> pd.DataFrame:
        """
        Generate LONG/SHORT signals from TEMA direction.
        
        Process:
        1. Calculate TEMA change (already done in TEMACalculator)
        2. Generate signals: LONG if rising, SHORT if falling
        3. Forward-fill signals (maintain position until next signal)
        
        Args:
            df: DataFrame with TEMA_change column
            
        Returns:
            DataFrame with Signal column (1 = LONG, -1 = SHORT)
        """
        df = df.copy()
        
        # Initialize signal column
        df['Signal'] = np.nan
        
        # Generate signals based on TEMA change
        # TEMA_change > 0: TEMA is rising → LONG
        # TEMA_change < 0: TEMA is falling → SHORT
        # TEMA_change == 0: No change → Keep previous signal
        
        df.loc[df['TEMA_change'] > 0, 'Signal'] = 1  # LONG
        df.loc[df['TEMA_change'] < 0, 'Signal'] = -1  # SHORT
        
        # Forward-fill signals to maintain position between signal changes
        # This ensures we're always in the market (either LONG or SHORT)
        df['Signal'] = df['Signal'].fillna(method='ffill')
        
        # For the first valid signal, if still NaN, default to LONG
        # This handles the initial warm-up period
        first_valid_idx = df['Signal'].first_valid_index()
        if first_valid_idx is not None and pd.isna(df.loc[first_valid_idx, 'Signal']):
            df.loc[first_valid_idx, 'Signal'] = 1
        
        df['Signal'] = df['Signal'].fillna(method='ffill')
        
        # Calculate signal changes (for position transitions)
        df['Signal_change'] = df['Signal'].diff()
        
        # Count signals
        long_signals = (df['Signal'] == 1).sum()
        short_signals = (df['Signal'] == -1).sum()
        signal_changes = (df['Signal_change'] != 0).sum()
        
        print(f"\nSignal Generation:")
        print(f"  LONG signals: {long_signals} days ({long_signals/len(df)*100:.1f}%)")
        print(f"  SHORT signals: {short_signals} days ({short_signals/len(df)*100:.1f}%)")
        print(f"  Signal changes (trades): {signal_changes}")
        
        return df


# ============================================================================
# BACKTEST ENGINE
# ============================================================================

class BacktestEngine:
    """
    Execute realistic backtest with proper execution flow.
    
    EXECUTION FLOW:
    ===============
    Day T (Signal Day):
      - Observe Close[T]
      - TEMA[T-1] and TEMA[T-2] are known
      - Generate Signal for T+1
    
    Day T+1 (Execution Day):
      - Execute at Open[T+1]
      - If signal changed: close old position, open new position
      - Calculate return: Open[T+1]/Open[T] - 1
    
    Day T+1 to T+N (Holding Days):
      - Calculate returns: Close[T]/Close[T-1] - 1
      - Accrue costs (commissions, borrow fees for shorts)
    
    TRANSACTION COSTS:
    ==================
    - Commission per trade (fixed + percentage)
    - Bid-ask spread
    - Market impact
    - Short borrow costs (for short positions)
    """
    
    def __init__(self, config: StrategyConfig):
        """
        Initialize backtest engine with strategy configuration.
        
        Args:
            config: StrategyConfig object with all parameters
        """
        self.config = config
        self.trades = []
        self.daily_stats = []
    
    def calculate_transaction_cost(self, 
                                   trade_value: float, 
                                   is_short: bool = False) -> float:
        """
        Calculate realistic transaction costs.
        
        Components:
        1. Fixed commission per trade
        2. Percentage commission
        3. Bid-ask spread cost
        4. Market impact cost
        
        Args:
            trade_value: Dollar value of trade (absolute)
            is_short: Whether this is a short sale (affects costs)
            
        Returns:
            Total transaction cost in dollars
        """
        abs_value = abs(trade_value)
        
        # Component 1: Commission (fixed + percentage)
        commission = (
            self.config.COMMISSION_PER_TRADE + 
            abs_value * self.config.COMMISSION_PCT
        )
        
        # Component 2: Bid-ask spread (paid on both entry and exit)
        spread_cost = abs_value * self.config.BID_ASK_SPREAD
        
        # Component 3: Market impact (price moves against us)
        impact_cost = abs_value * self.config.MARKET_IMPACT
        
        # Component 4: Additional costs for short sales
        short_premium = 0.0
        if is_short:
            # Shorts typically have higher costs (locate fees, etc.)
            short_premium = abs_value * 0.0001  # 1 basis point extra
        
        total_cost = commission + spread_cost + impact_cost + short_premium
        
        return total_cost
    
    def calculate_short_borrow_cost(self, 
                                   position_value: float, 
                                   days_held: int = 1) -> float:
        """
        Calculate cost to borrow shares for short position.
        
        Short borrow rate is annualized, so we prorate it daily.
        
        Args:
            position_value: Dollar value of short position
            days_held: Number of days position was held
            
        Returns:
            Borrow cost in dollars
        """
        daily_rate = self.config.SHORT_BORROW_RATE / 252
        return abs(position_value) * daily_rate * days_held
    
    def run_backtest(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Execute backtest with realistic transaction costs and execution.
        
        Process:
        ========
        1. Filter to strategy period (after TEMA warm-up)
        2. For each day with a signal:
           a. Check if position needs to change
           b. If yes: execute trades at Open[t+1]
           c. Calculate returns during holding period
           d. Apply transaction costs and borrow fees
        3. Track portfolio value over time
        
        Args:
            df: DataFrame with signals and prices
            
        Returns:
            DataFrame with backtest results and statistics
        """
        print(f"\n{'='*80}")
        print("EXECUTING BACKTEST")
        print(f"{'='*80}")
        
        # Filter to strategy period (after TEMA warm-up)
        strategy_start = pd.to_datetime(self.config.START_DATE)
        df_backtest = df[df['Date'] >= strategy_start].copy().reset_index(drop=True)
        
        if len(df_backtest) < 50:
            raise ValueError(
                f"Insufficient data for backtest: {len(df_backtest)} days"
            )
        
        print(f"\nBacktest Period:")
        print(f"  Start: {df_backtest['Date'].min().date()}")
        print(f"  End: {df_backtest['Date'].max().date()}")
        print(f"  Trading days: {len(df_backtest)}")
        
        # Initialize tracking
        capital = self.config.INITIAL_CAPITAL
        position = 0  # Current position: 1=LONG, -1=SHORT, 0=FLAT
        position_price = 0  # Price at which current position was entered
        position_date = None
        
        # Results storage
        df_backtest['Portfolio_Value'] = np.nan
        df_backtest['Strategy_Returns'] = 0.0
        df_backtest['Position'] = 0
        df_backtest['Transaction_Cost'] = 0.0
        df_backtest['Borrow_Cost'] = 0.0
        
        # Track costs
        total_transaction_costs = 0.0
        total_borrow_costs = 0.0
        num_trades = 0
        
        # MAIN BACKTEST LOOP
        for i in range(len(df_backtest)):
            current_date = df_backtest.loc[i, 'Date']
            current_signal = df_backtest.loc[i, 'Signal']
            
            # Skip if signal is not valid yet
            if pd.isna(current_signal):
                df_backtest.loc[i, 'Portfolio_Value'] = capital
                continue
            
            current_signal = int(current_signal)
            
            # === POSITION MANAGEMENT ===
            
            # Check if we need to change position
            position_change = False
            if i == 0:
                # First day: enter initial position
                position_change = True
            elif current_signal != position:
                # Signal changed: need to flip position
                position_change = True
            
            if position_change and i < len(df_backtest) - 1:
                # === EXECUTION AT OPEN[T+1] ===
                
                # Get execution price (next day's open)
                next_idx = i + 1
                execution_price = df_backtest.loc[next_idx, 'Open']
                
                if pd.isna(execution_price) or execution_price <= 0:
                    # Skip if execution price is invalid
                    df_backtest.loc[i, 'Portfolio_Value'] = capital
                    continue
                
                # Close old position (if exists)
                if position != 0 and position_price > 0:
                    # Calculate P&L from old position
                    if i > 0:
                        # Get previous day's open for return calculation
                        prev_open = df_backtest.loc[i, 'Open']
                        
                        if pd.notna(prev_open) and prev_open > 0:
                            # Return = Open[t+1]/Open[t] - 1
                            price_return = (execution_price / prev_open) - 1
                            
                            # Apply position direction
                            position_return = price_return * position
                            
                            # Calculate P&L
                            pnl = capital * position_return
                            capital += pnl
                            
                            # Transaction cost to close position
                            close_cost = self.calculate_transaction_cost(
                                capital * self.config.POSITION_SIZE,
                                is_short=(position == -1)
                            )
                            capital -= close_cost
                            total_transaction_costs += close_cost
                            
                            # Short borrow cost (if was short)
                            if position == -1 and position_date is not None:
                                days_held = (current_date - position_date).days
                                borrow_cost = self.calculate_short_borrow_cost(
                                    capital * self.config.POSITION_SIZE,
                                    days_held
                                )
                                capital -= borrow_cost
                                total_borrow_costs += borrow_cost
                                df_backtest.loc[i, 'Borrow_Cost'] = borrow_cost
                
                # Open new position
                # Transaction cost to open new position
                open_cost = self.calculate_transaction_cost(
                    capital * self.config.POSITION_SIZE,
                    is_short=(current_signal == -1)
                )
                capital -= open_cost
                total_transaction_costs += open_cost
                df_backtest.loc[i, 'Transaction_Cost'] = close_cost + open_cost if position != 0 else open_cost
                
                # Update position tracking
                position = current_signal
                position_price = execution_price
                position_date = current_date
                num_trades += 1
                
                # Record trade
                self.trades.append({
                    'Date': current_date,
                    'Execution_Date': df_backtest.loc[next_idx, 'Date'],
                    'Execution_Price': execution_price,
                    'Signal': 'LONG' if current_signal == 1 else 'SHORT',
                    'Position_Size': self.config.POSITION_SIZE,
                    'Cost': open_cost
                })
            
            # === HOLDING PERIOD RETURNS ===
            
            # During holding period, calculate Close-to-Close returns
            if i > 0 and position != 0:
                prev_close = df_backtest.loc[i-1, 'Close']
                curr_close = df_backtest.loc[i, 'Close']
                
                if pd.notna(prev_close) and pd.notna(curr_close) and prev_close > 0:
                    # Close-to-Close return
                    price_return = (curr_close / prev_close) - 1
                    
                    # Apply position direction
                    position_return = price_return * position
                    
                    # Calculate P&L
                    pnl = capital * position_return
                    capital += pnl
                    
                    # Daily short borrow cost (if short)
                    if position == -1:
                        daily_borrow = self.calculate_short_borrow_cost(
                            capital * self.config.POSITION_SIZE,
                            days_held=1
                        )
                        capital -= daily_borrow
                        total_borrow_costs += daily_borrow
                        df_backtest.loc[i, 'Borrow_Cost'] = daily_borrow
                    
                    # Store strategy return
                    df_backtest.loc[i, 'Strategy_Returns'] = position_return
            
            # Update tracking
            df_backtest.loc[i, 'Portfolio_Value'] = capital
            df_backtest.loc[i, 'Position'] = position
        
        # === BACKTEST SUMMARY ===
        
        final_value = df_backtest['Portfolio_Value'].iloc[-1]
        total_return = (final_value / self.config.INITIAL_CAPITAL) - 1
        
        print(f"\n{'='*80}")
        print("BACKTEST COMPLETE")
        print(f"{'='*80}")
        print(f"\nTrading Summary:")
        print(f"  Total trades: {num_trades}")
        print(f"  Transaction costs: ${total_transaction_costs:,.2f}")
        print(f"  Short borrow costs: ${total_borrow_costs:,.2f}")
        print(f"  Total costs: ${total_transaction_costs + total_borrow_costs:,.2f}")
        print(f"\nPerformance:")
        print(f"  Initial capital: ${self.config.INITIAL_CAPITAL:,.2f}")
        print(f"  Final value: ${final_value:,.2f}")
        print(f"  Total return: {total_return:.2%}")
        
        return df_backtest


# ============================================================================
# PERFORMANCE ANALYTICS
# ============================================================================

class PerformanceAnalyzer:
    """
    Calculate comprehensive performance metrics.
    
    METRICS CALCULATED:
    ===================
    Returns:
    - Total return
    - Annualized return (CAGR)
    - Monthly/yearly returns
    
    Risk:
    - Volatility (annualized)
    - Maximum drawdown
    - Downside deviation
    
    Risk-Adjusted:
    - Sharpe ratio
    - Sortino ratio
    - Calmar ratio
    
    Trade Statistics:
    - Win rate
    - Average win/loss
    - Profit factor
    """
    
    def __init__(self, config: StrategyConfig):
        """
        Initialize analyzer with strategy configuration.
        
        Args:
            config: StrategyConfig object
        """
        self.config = config
    
    def calculate_metrics(self, 
                         df: pd.DataFrame,
                         benchmark_df: Optional[pd.DataFrame] = None) -> Dict:
        """
        Calculate comprehensive performance metrics.
        
        Args:
            df: Backtest results DataFrame
            benchmark_df: Optional benchmark data for comparison
            
        Returns:
            Dictionary of performance metrics
        """
        print(f"\n{'='*80}")
        print("CALCULATING PERFORMANCE METRICS")
        print(f"{'='*80}")
        
        # Filter to valid returns
        returns = df['Strategy_Returns'].dropna()
        values = df['Portfolio_Value'].dropna()
        
        if len(returns) < 2:
            raise ValueError("Insufficient data for metrics calculation")
        
        # Time period
        start_date = df['Date'].min()
        end_date = df['Date'].max()
        days = len(returns)
        years = days / 252
        
        # === RETURN METRICS ===
        
        initial_value = self.config.INITIAL_CAPITAL
        final_value = values.iloc[-1]
        
        total_return = (final_value / initial_value) - 1
        
        # CAGR: Compound Annual Growth Rate
        cagr = (final_value / initial_value) ** (1 / years) - 1 if years > 0 else 0
        
        # === RISK METRICS ===
        
        # Volatility (annualized standard deviation)
        volatility = returns.std() * np.sqrt(252)
        
        # Maximum Drawdown
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        
        # Downside Deviation (for Sortino ratio)
        negative_returns = returns[returns < 0]
        downside_deviation = negative_returns.std() * np.sqrt(252) if len(negative_returns) > 0 else 0
        
        # === RISK-ADJUSTED METRICS ===
        
        # Sharpe Ratio: (Return - RiskFree) / Volatility
        sharpe_ratio = (cagr - self.config.RISK_FREE_RATE) / volatility if volatility > 0 else 0
        
        # Sortino Ratio: (Return - RiskFree) / Downside Deviation
        sortino_ratio = (cagr - self.config.RISK_FREE_RATE) / downside_deviation if downside_deviation > 0 else 0
        
        # Calmar Ratio: CAGR / Max Drawdown
        calmar_ratio = abs(cagr / max_drawdown) if max_drawdown != 0 else 0
        
        # === TRADE STATISTICS ===
        
        # Win rate
        winning_days = (returns > 0).sum()
        losing_days = (returns < 0).sum()
        win_rate = winning_days / len(returns) if len(returns) > 0 else 0
        
        # Average win/loss
        avg_win = returns[returns > 0].mean() if winning_days > 0 else 0
        avg_loss = returns[returns < 0].mean() if losing_days > 0 else 0
        
        # Profit factor
        gross_profit = returns[returns > 0].sum()
        gross_loss = abs(returns[returns < 0].sum())
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else 0
        
        # === BENCHMARK COMPARISON ===
        
        benchmark_metrics = {}
        if benchmark_df is not None:
            bench_returns = benchmark_df['Close'].pct_change().dropna()
            bench_cumulative = (1 + bench_returns).cumprod()
            
            bench_total_return = bench_cumulative.iloc[-1] - 1
            bench_cagr = (1 + bench_total_return) ** (1 / years) - 1 if years > 0 else 0
            bench_volatility = bench_returns.std() * np.sqrt(252)
            bench_sharpe = (bench_cagr - self.config.RISK_FREE_RATE) / bench_volatility if bench_volatility > 0 else 0
            
            bench_running_max = bench_cumulative.expanding().max()
            bench_drawdown = (bench_cumulative - bench_running_max) / bench_running_max
            bench_max_dd = bench_drawdown.min()
            
            benchmark_metrics = {
                'total_return': bench_total_return,
                'cagr': bench_cagr,
                'volatility': bench_volatility,
                'sharpe_ratio': bench_sharpe,
                'max_drawdown': bench_max_dd
            }
        
        # Compile all metrics
        metrics = {
            'start_date': start_date,
            'end_date': end_date,
            'days': days,
            'years': years,
            'total_return': total_return,
            'cagr': cagr,
            'volatility': volatility,
            'max_drawdown': max_drawdown,
            'downside_deviation': downside_deviation,
            'sharpe_ratio': sharpe_ratio,
            'sortino_ratio': sortino_ratio,
            'calmar_ratio': calmar_ratio,
            'win_rate': win_rate,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'profit_factor': profit_factor,
            'winning_days': winning_days,
            'losing_days': losing_days,
            'benchmark': benchmark_metrics
        }
        
        # Print metrics
        self._print_metrics(metrics)
        
        return metrics
    
    def _print_metrics(self, metrics: Dict) -> None:
        """
        Print formatted metrics table.
        
        Args:
            metrics: Dictionary of performance metrics
        """
        print(f"\nPeriod: {metrics['start_date'].date()} to {metrics['end_date'].date()}")
        print(f"Trading days: {metrics['days']} ({metrics['years']:.2f} years)")
        
        print(f"\n{'='*80}")
        print("STRATEGY PERFORMANCE")
        print(f"{'='*80}")
        
        print(f"\nReturn Metrics:")
        print(f"  Total Return:        {metrics['total_return']:>10.2%}")
        print(f"  CAGR:                {metrics['cagr']:>10.2%}")
        
        print(f"\nRisk Metrics:")
        print(f"  Volatility:          {metrics['volatility']:>10.2%}")
        print(f"  Max Drawdown:        {metrics['max_drawdown']:>10.2%}")
        print(f"  Downside Deviation:  {metrics['downside_deviation']:>10.2%}")
        
        print(f"\nRisk-Adjusted Metrics:")
        print(f"  Sharpe Ratio:        {metrics['sharpe_ratio']:>10.2f}")
        print(f"  Sortino Ratio:       {metrics['sortino_ratio']:>10.2f}")
        print(f"  Calmar Ratio:        {metrics['calmar_ratio']:>10.2f}")
        
        print(f"\nTrade Statistics:")
        print(f"  Win Rate:            {metrics['win_rate']:>10.2%}")
        print(f"  Winning Days:        {metrics['winning_days']:>10d}")
        print(f"  Losing Days:         {metrics['losing_days']:>10d}")
        print(f"  Average Win:         {metrics['avg_win']:>10.4f}")
        print(f"  Average Loss:        {metrics['avg_loss']:>10.4f}")
        print(f"  Profit Factor:       {metrics['profit_factor']:>10.2f}")
        
        if metrics['benchmark']:
            bench = metrics['benchmark']
            print(f"\n{'='*80}")
            print(f"BENCHMARK COMPARISON ({self.config.BENCHMARK_TICKER})")
            print(f"{'='*80}")
            print(f"\nBenchmark Performance:")
            print(f"  Total Return:        {bench['total_return']:>10.2%}")
            print(f"  CAGR:                {bench['cagr']:>10.2%}")
            print(f"  Volatility:          {bench['volatility']:>10.2%}")
            print(f"  Sharpe Ratio:        {bench['sharpe_ratio']:>10.2f}")
            print(f"  Max Drawdown:        {bench['max_drawdown']:>10.2%}")
            
            print(f"\nRelative Performance:")
            print(f"  Excess Return:       {metrics['cagr'] - bench['cagr']:>10.2%}")
            print(f"  Sharpe Difference:   {metrics['sharpe_ratio'] - bench['sharpe_ratio']:>10.2f}")


# ============================================================================
# VISUALIZATION MODULE
# ============================================================================

class Visualizer:
    """
    Create comprehensive visualizations for strategy analysis.
    
    CHARTS INCLUDED:
    ================
    1. Price and TEMA: Shows close price with TEMA overlay
    2. Equity Curve: Strategy performance vs benchmark
    3. Drawdown: Underwater equity chart
    4. Position Changes: Long/short position timeline
    5. Returns Distribution: Histogram of daily returns
    """
    
    def __init__(self, config: StrategyConfig):
        """
        Initialize visualizer with strategy configuration.
        
        Args:
            config: StrategyConfig object
        """
        self.config = config
    
    def plot_comprehensive_analysis(self,
                                    df: pd.DataFrame,
                                    metrics: Dict,
                                    benchmark_df: Optional[pd.DataFrame] = None) -> None:
        """
        Create comprehensive visualization dashboard.
        
        Args:
            df: Backtest results DataFrame
            metrics: Performance metrics dictionary
            benchmark_df: Optional benchmark data
        """
        # Set style
        plt.style.use('seaborn-v0_8-darkgrid')
        sns.set_palette("husl")
        
        # Create figure with subplots
        fig = plt.figure(figsize=(20, 14))
        gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)
        
        # Filter to strategy period
        strategy_start = pd.to_datetime(self.config.START_DATE)
        df_plot = df[df['Date'] >= strategy_start].copy()
        
        # ====================================================================
        # CHART 1: PRICE AND TEMA
        # ====================================================================
        ax1 = fig.add_subplot(gs[0, :])
        
        # Plot close price
        ax1.plot(df_plot['Date'], df_plot['Close'],
                label='Close Price', color='black', linewidth=1.5, alpha=0.7)
        
        # Plot TEMA
        ax1.plot(df_plot['Date'], df_plot['TEMA_lag'],
                label=f'TEMA({self.config.TEMA_PERIOD})',
                color='blue', linewidth=2.5)
        
        # Highlight LONG periods
        long_periods = df_plot[df_plot['Signal'] == 1]
        ax1.scatter(long_periods['Date'], long_periods['Close'],
                   color='green', alpha=0.3, s=10, label='LONG')
        
        # Highlight SHORT periods
        short_periods = df_plot[df_plot['Signal'] == -1]
        ax1.scatter(short_periods['Date'], short_periods['Close'],
                   color='red', alpha=0.3, s=10, label='SHORT')
        
        ax1.set_title(f'{self.config.TICKER} Price and TEMA({self.config.TEMA_PERIOD})',
                     fontsize=16, fontweight='bold')
        ax1.set_ylabel('Price ($)', fontsize=12)
        ax1.legend(loc='best', fontsize=10)
        ax1.grid(True, alpha=0.3)
        
        # ====================================================================
        # CHART 2: EQUITY CURVE
        # ====================================================================
        ax2 = fig.add_subplot(gs[1, :])
        
        # Plot strategy equity
        strategy_equity = df_plot['Portfolio_Value'].dropna()
        ax2.plot(df_plot.loc[strategy_equity.index, 'Date'],
                strategy_equity,
                label='Strategy', color='blue', linewidth=2.5)
        
        # Plot benchmark if available
        if benchmark_df is not None:
            # Align benchmark to strategy dates
            bench_aligned = benchmark_df[
                (benchmark_df['Date'] >= df_plot['Date'].min()) &
                (benchmark_df['Date'] <= df_plot['Date'].max())
            ].copy()
            
            if len(bench_aligned) > 0:
                # Calculate benchmark equity
                bench_returns = bench_aligned['Close'].pct_change()
                bench_equity = self.config.INITIAL_CAPITAL * (1 + bench_returns).cumprod()
                
                ax2.plot(bench_aligned['Date'], bench_equity,
                        label=f'{self.config.BENCHMARK_TICKER} (Benchmark)',
                        color='orange', linewidth=2, alpha=0.7)
        
        ax2.set_title('Equity Curve: Strategy vs Benchmark',
                     fontsize=16, fontweight='bold')
        ax2.set_ylabel('Portfolio Value ($)', fontsize=12)
        ax2.legend(loc='best', fontsize=10)
        ax2.grid(True, alpha=0.3)
        
        # Add performance annotation
        final_value = strategy_equity.iloc[-1]
        total_return = (final_value / self.config.INITIAL_CAPITAL - 1) * 100
        
        textstr = f'Final Value: ${final_value:,.0f}\nTotal Return: {total_return:+.1f}%'
        ax2.text(0.02, 0.98, textstr,
                transform=ax2.transAxes,
                fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # ====================================================================
        # CHART 3: DRAWDOWN
        # ====================================================================
        ax3 = fig.add_subplot(gs[2, 0])
        
        # Calculate drawdown
        returns = df_plot['Strategy_Returns'].dropna()
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        
        # Plot drawdown
        drawdown_dates = df_plot.loc[drawdown.index, 'Date']
        ax3.fill_between(drawdown_dates, drawdown * 100, 0,
                        color='red', alpha=0.3)
        ax3.plot(drawdown_dates, drawdown * 100,
                color='darkred', linewidth=1.5)
        
        ax3.set_title('Strategy Drawdown',
                     fontsize=14, fontweight='bold')
        ax3.set_ylabel('Drawdown (%)', fontsize=12)
        ax3.set_xlabel('Date', fontsize=12)
        ax3.grid(True, alpha=0.3)
        
        # Add max drawdown annotation
        max_dd = drawdown.min() * 100
        ax3.text(0.02, 0.02, f'Max DD: {max_dd:.2f}%',
                transform=ax3.transAxes,
                fontsize=10,
                verticalalignment='bottom',
                bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))
        
        # ====================================================================
        # CHART 4: POSITION CHANGES
        # ====================================================================
        ax4 = fig.add_subplot(gs[2, 1])
        
        # Plot position over time
        position_data = df_plot[['Date', 'Position']].dropna()
        
        # Color-code positions
        for i in range(len(position_data) - 1):
            date_start = position_data.iloc[i]['Date']
            date_end = position_data.iloc[i + 1]['Date']
            position = position_data.iloc[i]['Position']
            
            color = 'green' if position == 1 else 'red' if position == -1 else 'gray'
            ax4.axvspan(date_start, date_end, alpha=0.3, color=color)
        
        ax4.plot(position_data['Date'], position_data['Position'],
                color='black', linewidth=2, marker='o', markersize=3)
        
        ax4.set_title('Position Timeline',
                     fontsize=14, fontweight='bold')
        ax4.set_ylabel('Position (1=LONG, -1=SHORT)', fontsize=12)
        ax4.set_xlabel('Date', fontsize=12)
        ax4.set_yticks([-1, 0, 1])
        ax4.set_yticklabels(['SHORT', 'FLAT', 'LONG'])
        ax4.grid(True, alpha=0.3)
        
        # ====================================================================
        # CHART 5: RETURNS DISTRIBUTION
        # ====================================================================
        ax5 = fig.add_subplot(gs[3, 0])
        
        # Histogram of returns
        returns_clean = returns[returns != 0]
        
        ax5.hist(returns_clean * 100, bins=50,
                color='blue', alpha=0.6, edgecolor='black')
        
        ax5.axvline(x=0, color='black', linestyle='--', linewidth=2)
        ax5.axvline(x=returns_clean.mean() * 100,
                   color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {returns_clean.mean()*100:.3f}%')
        
        ax5.set_title('Daily Returns Distribution',
                     fontsize=14, fontweight='bold')
        ax5.set_xlabel('Daily Return (%)', fontsize=12)
        ax5.set_ylabel('Frequency', fontsize=12)
        ax5.legend(loc='best', fontsize=10)
        ax5.grid(True, alpha=0.3)
        
        # ====================================================================
        # CHART 6: PERFORMANCE METRICS
        # ====================================================================
        ax6 = fig.add_subplot(gs[3, 1])
        ax6.axis('off')
        
        # Create metrics text
        metrics_text = f"""
PERFORMANCE METRICS
{'='*40}

Period: {metrics['start_date'].date()} to {metrics['end_date'].date()}
Trading Days: {metrics['days']} ({metrics['years']:.2f} years)

RETURNS:
  Total Return:    {metrics['total_return']:>8.2%}
  CAGR:            {metrics['cagr']:>8.2%}

RISK:
  Volatility:      {metrics['volatility']:>8.2%}
  Max Drawdown:    {metrics['max_drawdown']:>8.2%}

RISK-ADJUSTED:
  Sharpe Ratio:    {metrics['sharpe_ratio']:>8.2f}
  Sortino Ratio:   {metrics['sortino_ratio']:>8.2f}
  Calmar Ratio:    {metrics['calmar_ratio']:>8.2f}

TRADE STATS:
  Win Rate:        {metrics['win_rate']:>8.2%}
  Winning Days:    {metrics['winning_days']:>8d}
  Losing Days:     {metrics['losing_days']:>8d}
  Profit Factor:   {metrics['profit_factor']:>8.2f}
        """
        
        ax6.text(0.1, 0.95, metrics_text,
                transform=ax6.transAxes,
                fontsize=10,
                verticalalignment='top',
                fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        # Main title
        fig.suptitle(
            f'TEMA({self.config.TEMA_PERIOD}) Long-Short Strategy - {self.config.TICKER}',
            fontsize=20, fontweight='bold', y=0.995
        )
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n{'='*80}")
        print("VISUALIZATION COMPLETE")
        print(f"{'='*80}")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function for TEMA strategy.
    
    Process:
    1. Initialize configuration
    2. Fetch data
    3. Calculate TEMA
    4. Generate signals
    5. Run backtest
    6. Calculate metrics
    7. Create visualizations
    """
    
    print("="*80)
    print("TEMA LONG-SHORT TRADING STRATEGY")
    print("="*80)
    
    # ====================================================================
    # STEP 1: INITIALIZE CONFIGURATION
    # ====================================================================
    
    config = StrategyConfig()
    
    print(f"\nStrategy Configuration:")
    print(f"  Ticker: {config.TICKER}")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  TEMA Period: {config.TEMA_PERIOD}")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Benchmark: {config.BENCHMARK_TICKER}")
    
    # ====================================================================
    # STEP 2: FETCH DATA
    # ====================================================================
    
    print(f"\n{'='*80}")
    print("STEP 1: DATA ACQUISITION")
    print(f"{'='*80}")
    
    data_fetcher = DataFetcher()
    
    # Fetch main ticker data
    df = data_fetcher.fetch_ohlcv_data(
        config.TICKER,
        config.START_DATE,
        config.END_DATE
    )
    
    # Fetch benchmark data
    try:
        df_benchmark = data_fetcher.fetch_ohlcv_data(
            config.BENCHMARK_TICKER,
            config.START_DATE,
            config.END_DATE
        )
        print(f"  ✓ Benchmark {config.BENCHMARK_TICKER} data fetched")
    except Exception as e:
        print(f"  ! Warning: Could not fetch benchmark data: {e}")
        df_benchmark = None
    
    # ====================================================================
    # STEP 3: CALCULATE TEMA
    # ====================================================================
    
    print(f"\n{'='*80}")
    print("STEP 2: TEMA CALCULATION")
    print(f"{'='*80}")
    
    tema_calculator = TEMACalculator()
    df = tema_calculator.calculate_tema_with_validation(df, config.TEMA_PERIOD)
    
    # ====================================================================
    # STEP 4: GENERATE SIGNALS
    # ====================================================================
    
    print(f"\n{'='*80}")
    print("STEP 3: SIGNAL GENERATION")
    print(f"{'='*80}")
    
    signal_generator = SignalGenerator()
    df = signal_generator.generate_signals(df)
    
    # ====================================================================
    # STEP 5: RUN BACKTEST
    # ====================================================================
    
    print(f"\n{'='*80}")
    print("STEP 4: BACKTEST EXECUTION")
    print(f"{'='*80}")
    
    backtest_engine = BacktestEngine(config)
    df_results = backtest_engine.run_backtest(df)
    
    # ====================================================================
    # STEP 6: CALCULATE METRICS
    # ====================================================================
    
    print(f"\n{'='*80}")
    print("STEP 5: PERFORMANCE ANALYSIS")
    print(f"{'='*80}")
    
    analyzer = PerformanceAnalyzer(config)
    metrics = analyzer.calculate_metrics(df_results, df_benchmark)
    
    # ====================================================================
    # STEP 7: CREATE VISUALIZATIONS
    # ====================================================================
    
    print(f"\n{'='*80}")
    print("STEP 6: VISUALIZATION")
    print(f"{'='*80}")
    
    visualizer = Visualizer(config)
    visualizer.plot_comprehensive_analysis(df_results, metrics, df_benchmark)
    
    # ====================================================================
    # STEP 8: EXPORT RESULTS
    # ====================================================================
    
    print(f"\n{'='*80}")
    print("EXPORTING RESULTS")
    print(f"{'='*80}")
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Export full results
    output_file = f"tema_strategy_results_{config.TICKER}_{timestamp}.csv"
    df_results.to_csv(output_file, index=False)
    print(f"  ✓ Results exported to: {output_file}")
    
    # Export trades
    if backtest_engine.trades:
        trades_df = pd.DataFrame(backtest_engine.trades)
        trades_file = f"tema_strategy_trades_{config.TICKER}_{timestamp}.csv"
        trades_df.to_csv(trades_file, index=False)
        print(f"  ✓ Trades exported to: {trades_file}")
    
    print(f"\n{'='*80}")
    print("EXECUTION COMPLETE")
    print(f"{'='*80}\n")
    
    return df_results, metrics, config


if __name__ == "__main__":
    results, metrics, config = main()